In [3]:
# ===== BOOTSTRAP: run first after every Colab start/restart =====
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

# 1) Reconnect Google Drive (permanent storage)
drive.mount('/content/drive')

# 2) Load secrets from the Colab Secrets panel into environment variables
SECRETS = ["GITHUB_USERNAME", "GITHUB_TOKEN", "GIT_EMAIL", "DAGSHUB_USERNAME",
           "DAGSHUB_TOKEN", "KAGGLE_API_TOKEN"]
missing = []
for k in SECRETS:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        missing.append(k)
print(f"Secrets loaded: {len(SECRETS)-len(missing)}/{len(SECRETS)}  missing: {missing or 'none'}")

# 3) Project paths
PROJECT     = "23L-0951_A2_Ops"
DRIVE       = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE / PROJECT              # git repo (code, notebooks, small files)
WORK_DIR    = DRIVE / f"{PROJECT}_work"    # big files kept OUT of git
LOCAL_DATA  = Path("/content/data")        # fast local copy, rebuilt each session
for p in (WORK_DIR, LOCAL_DATA):
    p.mkdir(parents=True, exist_ok=True)
os.environ.update(PROJECT_DIR=str(PROJECT_DIR), WORK_DIR=str(WORK_DIR),
                  LOCAL_DATA=str(LOCAL_DATA))
os.environ["DVC_NO_ANALYTICS"] = "1"

def sh(cmd, cwd=None):
    """Run a shell command and print its output. Returns the exit code."""
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

# 4) Git settings: one credential helper per website, tokens read from env vars.
#    GitHub gets the GitHub token, DagsHub gets the DagsHub token.
#    Tokens are never saved inside the repo or in remote URLs.
gh_helper = '!f() { echo "username=${GITHUB_USERNAME}"; echo "password=${GITHUB_TOKEN}"; }; f'
dh_helper = '!f() { echo "username=${DAGSHUB_USERNAME}"; echo "password=${DAGSHUB_TOKEN}"; }; f'
subprocess.run(["git", "config", "--global", "--unset-all", "credential.helper"])  # remove old generic helper
for key, val in [("credential.https://github.com.helper", gh_helper),
                 ("credential.https://dagshub.com.helper", dh_helper),
                 ("user.name", "Zayan"),
                 ("user.email", os.environ.get("GIT_EMAIL", "")),
                 ("init.defaultBranch", "main")]:
    subprocess.run(["git", "config", "--global", key, val], check=True)

# 5) Clone the repo into Drive the first time; afterwards just pull updates
if not (PROJECT_DIR / ".git").exists():
    sh(f'git clone https://github.com/{os.environ["GITHUB_USERNAME"]}/{PROJECT}.git "{PROJECT_DIR}"')
if not (PROJECT_DIR / ".git").exists():
    raise SystemExit("Clone failed: check GITHUB_USERNAME / GITHUB_TOKEN / repo name.")
os.chdir(PROJECT_DIR)
sh("rm -f .git/index.lock")      # clears a leftover lock from a dropped session
sh("git pull --ff-only")

# 6) Reinstall pinned packages
if (PROJECT_DIR / "requirements-colab.txt").exists():
    sh("pip install -q -r requirements-colab.txt")

# 7) Helper to save a step: copy notebook into repo, commit, push
def save_step(message, notebook_name=None):
    if notebook_name:
        src = DRIVE / "Colab Notebooks" / notebook_name
        (PROJECT_DIR / "notebooks").mkdir(exist_ok=True)
        if src.exists():
            shutil.copy(src, PROJECT_DIR / "notebooks" / notebook_name)
        else:
            print("Notebook not found:", src)
    sh("git add -A")
    sh(f'git commit -m "{message}"')
    sh("git push origin main")
    sh("git log --oneline -1")

print("Bootstrap done. Working directory:", os.getcwd())

Mounted at /content/drive
Secrets loaded: 6/6  missing: none
Already up to date.
Bootstrap done. Working directory: /content/drive/MyDrive/23L-0951_A2_Ops


In [4]:
import mlflow, mlflow.keras, keras, numpy as np, matplotlib.pyplot as plt

# Fashion-MNIST: 70k small grayscale clothing images, 10 classes, built into Keras
(x_tr, y_tr), (x_te, y_te) = keras.datasets.fashion_mnist.load_data()
x_tr, x_te = x_tr.astype("float32") / 255.0, x_te.astype("float32") / 255.0

class MlflowEpochLogger(keras.callbacks.Callback):
    """Manual logging: send this epoch's metrics to MLflow, with the epoch as the step."""
    def on_epoch_end(self, epoch, logs=None):
        mlflow.log_metrics({k: float(v) for k, v in (logs or {}).items()}, step=epoch)

def train_toy(lr, epochs=5, units=64, batch=128, run_name=None):
    keras.utils.set_random_seed(42)            # same start for both runs: only LR differs
    model = keras.Sequential([
        keras.Input((28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(units, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    with mlflow.start_run(run_name=run_name or f"toy_lr_{lr}") as run:
        mlflow.set_tags({"stage": "tool-lab", "dataset": "fashion_mnist", "student": "23L-0951"})
        mlflow.log_params({"learning_rate": lr, "epochs": epochs, "units": units,
                           "batch_size": batch, "optimizer": "adam"})
        hist = model.fit(x_tr, y_tr, validation_split=0.1, epochs=epochs, batch_size=batch,
                         callbacks=[MlflowEpochLogger()], verbose=2)
        test_loss, test_acc = model.evaluate(x_te, y_te, verbose=0)
        mlflow.log_metrics({"test_loss": test_loss, "test_accuracy": test_acc})

        fig, ax = plt.subplots(figsize=(6, 4))      # one plot, logged as an artifact
        ax.plot(hist.history["loss"], label="train loss")
        ax.plot(hist.history["val_loss"], label="val loss")
        ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
        ax.set_title(f"Fashion-MNIST toy, lr={lr}")
        mlflow.log_figure(fig, "loss_curve.png"); plt.close(fig)

        mlflow.keras.log_model(model, name="model")  # MLflow 3 style
        print(f"run_id={run.info.run_id}  test_accuracy={test_acc:.4f}")
    return run.info.run_id

print("Ready. Train:", x_tr.shape, "Test:", x_te.shape)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Ready. Train: (60000, 28, 28) Test: (10000, 28, 28)


In [5]:
from pathlib import Path
LOCAL_MLFLOW = Path("/content/mlflow_local"); LOCAL_MLFLOW.mkdir(exist_ok=True)
LOCAL_URI = f"sqlite:///{LOCAL_MLFLOW}/mlflow.db"
mlflow.set_tracking_uri(LOCAL_URI)

EXP_LOCAL = "23L-0951_A2_Ops_toy_local"
if mlflow.get_experiment_by_name(EXP_LOCAL) is None:
    mlflow.create_experiment(EXP_LOCAL, artifact_location=(LOCAL_MLFLOW / "artifacts").as_uri())
mlflow.set_experiment(EXP_LOCAL)

for lr in [1e-3, 1e-2]:
    train_toy(lr)

2026/11/04 17:14:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/04 17:14:03 INFO mlflow.store.db.utils: Updating database tables


Epoch 1/5
422/422 - 4s - 10ms/step - accuracy: 0.7978 - loss: 0.6001 - val_accuracy: 0.8418 - val_loss: 0.4488
Epoch 2/5
422/422 - 0s - 838us/step - accuracy: 0.8513 - loss: 0.4308 - val_accuracy: 0.8543 - val_loss: 0.4087
Epoch 3/5
422/422 - 0s - 902us/step - accuracy: 0.8638 - loss: 0.3909 - val_accuracy: 0.8630 - val_loss: 0.3889
Epoch 4/5
422/422 - 0s - 918us/step - accuracy: 0.8711 - loss: 0.3669 - val_accuracy: 0.8652 - val_loss: 0.3778
Epoch 5/5
422/422 - 0s - 824us/step - accuracy: 0.8771 - loss: 0.3489 - val_accuracy: 0.8698 - val_loss: 0.3670


2026/11/04 17:14:16 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


run_id=51b13de0635b4eefa0464d58fbab96e6  test_accuracy=0.8609
Epoch 1/5
422/422 - 2s - 5ms/step - accuracy: 0.8166 - loss: 0.5140 - val_accuracy: 0.8535 - val_loss: 0.4056
Epoch 2/5
422/422 - 0s - 867us/step - accuracy: 0.8547 - loss: 0.3996 - val_accuracy: 0.8665 - val_loss: 0.3725
Epoch 3/5
422/422 - 0s - 751us/step - accuracy: 0.8660 - loss: 0.3700 - val_accuracy: 0.8682 - val_loss: 0.3731
Epoch 4/5
422/422 - 0s - 753us/step - accuracy: 0.8715 - loss: 0.3526 - val_accuracy: 0.8613 - val_loss: 0.3958
Epoch 5/5
422/422 - 0s - 722us/step - accuracy: 0.8744 - loss: 0.3407 - val_accuracy: 0.8683 - val_loss: 0.3873


2026/11/04 17:14:29 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


run_id=2485acc5fc9d45039afe5eaa313d0246  test_accuracy=0.8547


In [10]:
import subprocess, time
from google.colab import output

PORT = 5001   # fresh port, so the old server stuck on 5000 doesn't matter

ui = subprocess.Popen(
    ["mlflow", "ui", "--backend-store-uri", LOCAL_URI, "--host", "0.0.0.0", "--port", str(PORT),
     "--allowed-hosts", "*", "--cors-allowed-origins", "*"],
    stdout=open("/content/mlflow_ui.log", "w"), stderr=subprocess.STDOUT)
time.sleep(12)
print(open("/content/mlflow_ui.log").read()[-600:])
output.serve_kernel_port_as_window(PORT, path="/")

20:55 INFO mlflow.server.jobs.utils: Registered online_scoring_scheduler periodic task (runs every 1 minute)
2026/11/04 17:20:55 INFO mlflow.server.jobs.utils: Registered trace_archival_scheduler periodic task (polls every 1 minute and no-ops when trace archival is disabled or unconfigured)
2026/11/04 17:20:55 INFO mlflow.server.jobs.utils: Registered issue_detection_scheduler periodic task (polls every 15 minutes and no-ops when no experiment has a detection schedule)
2026/11/04 17:20:55 INFO mlflow.server.jobs.utils: SQL trace rollup scheduler was not registered because rollups are disabled

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [7]:
# 1) What did the MLflow server say? (the real error is in here)
print(open("/content/mlflow_ui.log").read()[-3000:])

# 2) Are the runs actually stored? (asks the database directly, bypassing the UI)
print(mlflow.search_runs(experiment_names=[EXP_LOCAL])[
    ["run_id", "status", "params.learning_rate", "metrics.test_accuracy"]])

52652 - "GET /static-files/static/js/9867.768a39b1.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52644 - "GET /static-files/static/js/2369.415f0230.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52656 - "GET /static-files/static/js/3121.3be60de2.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52662 - "GET /static-files/static/js/8484.db1876bb.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52668 - "GET /static-files/static/js/4751.94bc6db7.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52672 - "GET /static-files/static/js/8739.c191b2a3.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52674 - "GET /static-files/static/js/3793.0117dc93.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52688 - "GET /static-files/static/js/5343.bbb1f624.chunk.js HTTP/1.1" 200 OK
2026/11/04 17:16:26 INFO:     127.0.0.1:52704 - "GET /static-files/static/js/3569.b70453bd.chunk.js HTTP

In [11]:
# MLflow reads these three standard environment variables automatically.
# Values come from Colab Secrets, so no token is written in code.
os.environ["MLFLOW_TRACKING_URI"]      = f"https://dagshub.com/{os.environ['DAGSHUB_USERNAME']}/{PROJECT}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["DAGSHUB_USERNAME"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["DAGSHUB_TOKEN"]
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])   # switch from local SQLite to DagsHub

mlflow.set_experiment("23L-0951_A2_Ops_toy")
for lr in [1e-3, 1e-2]:
    train_toy(lr, run_name=f"dagshub_toy_lr_{lr}")

print("\nMLflow UI on DagsHub:", os.environ["MLFLOW_TRACKING_URI"])

2026/11/04 17:30:59 INFO mlflow.tracking.fluent: Experiment with name '23L-0951_A2_Ops_toy' does not exist. Creating a new experiment.


Epoch 1/5
422/422 - 2s - 5ms/step - accuracy: 0.7978 - loss: 0.6001 - val_accuracy: 0.8418 - val_loss: 0.4488
Epoch 2/5
422/422 - 0s - 1ms/step - accuracy: 0.8513 - loss: 0.4308 - val_accuracy: 0.8543 - val_loss: 0.4087
Epoch 3/5
422/422 - 4s - 9ms/step - accuracy: 0.8638 - loss: 0.3909 - val_accuracy: 0.8630 - val_loss: 0.3889
Epoch 4/5
422/422 - 0s - 1ms/step - accuracy: 0.8711 - loss: 0.3669 - val_accuracy: 0.8652 - val_loss: 0.3778
Epoch 5/5
422/422 - 0s - 1ms/step - accuracy: 0.8771 - loss: 0.3489 - val_accuracy: 0.8698 - val_loss: 0.3670


2026/11/04 17:31:23 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


run_id=6734379d88e14987a9ef0a5a5f4a5fc3  test_accuracy=0.8609
🏃 View run dagshub_toy_lr_0.001 at: https://dagshub.com/bhatti-zayan/23L-0951_A2_Ops.mlflow/#/experiments/0/runs/6734379d88e14987a9ef0a5a5f4a5fc3
🧪 View experiment at: https://dagshub.com/bhatti-zayan/23L-0951_A2_Ops.mlflow/#/experiments/0
Epoch 1/5
422/422 - 2s - 5ms/step - accuracy: 0.8166 - loss: 0.5140 - val_accuracy: 0.8535 - val_loss: 0.4056
Epoch 2/5
422/422 - 0s - 1ms/step - accuracy: 0.8547 - loss: 0.3996 - val_accuracy: 0.8665 - val_loss: 0.3725
Epoch 3/5
422/422 - 0s - 1ms/step - accuracy: 0.8660 - loss: 0.3700 - val_accuracy: 0.8682 - val_loss: 0.3731
Epoch 4/5
422/422 - 3s - 6ms/step - accuracy: 0.8715 - loss: 0.3526 - val_accuracy: 0.8613 - val_loss: 0.3958
Epoch 5/5
422/422 - 0s - 995us/step - accuracy: 0.8744 - loss: 0.3407 - val_accuracy: 0.8683 - val_loss: 0.3873


2026/11/04 17:31:43 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


run_id=024e7e0c9ad74e448e8744ecf0f06fd7  test_accuracy=0.8547
🏃 View run dagshub_toy_lr_0.01 at: https://dagshub.com/bhatti-zayan/23L-0951_A2_Ops.mlflow/#/experiments/0/runs/024e7e0c9ad74e448e8744ecf0f06fd7
🧪 View experiment at: https://dagshub.com/bhatti-zayan/23L-0951_A2_Ops.mlflow/#/experiments/0

MLflow UI on DagsHub: https://dagshub.com/bhatti-zayan/23L-0951_A2_Ops.mlflow
